In [12]:
import json
import os
from dotenv import load_dotenv
from groq import Groq

In [13]:
load_dotenv()

True

In [14]:
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [15]:
try:
    with open("webscraping_digitimes_supply_chain.json", "r", encoding="utf-8") as f:
        scraped_stories = json.load(f)
    print(f"Successfully loaded {len(scraped_stories)} articles from the raw scraper file!")
except FileNotFoundError:
    print("Error: Could not find 'digitimes_supply_chain_raw.json'. Make sure it's in this folder!")
    scraped_stories = []

Successfully loaded 29 articles from the raw scraper file!


In [16]:
def extract_entities_with_groq(articles):
    standardised_data = []
    
    system_instruction = """
    You should think like an expert data analyst. Analyse the tech headline and return a valid JSON object.
    Do not include any markdown format code blocks, notes, or conversation. 
    Return only raw JSON matching this schema structure:
    {"company": "Name", "country_of_impact": "Country", "technology": "Tech Sector", "supply_chain_risk_level": "Low/Medium/High"}
    """
    
    for article in articles:
        user_prompt = f"Analyse this article headline: '{article['title']}'"
        
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                response_format={ "type": "json_object" },
                messages=[
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.0
            )
            
            # Parse the text JSON response into a Python dictionary
            llm_output = json.loads(response.choices[0].message.content)
            
            # Merge article data with the extracted attributes
            enriched_item = {
                "original_title": article['title'],
                "url": article['link'],
                "extracted_company": llm_output.get("company", "Unknown"),
                "primary_country": llm_output.get("country_of_impact", "Unknown"),
                "technology_sector": llm_output.get("technology", "Unknown"),
                "risk_level": llm_output.get("supply_chain_risk_level", "Low")
            }
            standardised_data.append(enriched_item)
            
        except Exception as e:
            print(f"Error processing article: {article['title']}. Error: {e}")
            
    return standardised_data

final_structured_data = extract_entities_with_groq(scraped_stories)

with open("standardised_entities.json", "w", encoding="utf-8") as f:
    json.dump(final_structured_data, f, ensure_ascii=False, indent=2)

print(f"Success! data saved to standardised_entities.json")

Success! data saved to standardised_entities.json
